# Phase 2 with Pinecone & GPT-3

### Step 1: Set up Pinecone

In [ ]:
!pip install pinecone
!pip install python-dotenv

  Using cached orjson-3.11.4-cp313-cp313-macosx_15_0_arm64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl.metadata (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 9.9 MB/s  0:00:00
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl (208 kB)
Using cached orjson-3.11.4-cp313-cp313-macosx_15_0_arm64.whl (128 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pinecone]5/6 [pinecone]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


### Step 2: Create/Setup/Connect a Pinecone Index

In [13]:
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

# Load .env
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV", "us-east-1")  # default to us-east-1

# Create Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Index name
index_name = "youtube-chunks"

# Create index if it doesn't exist
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region=PINECONE_ENV
        )
    )

# Connect to the index
index = pc.Index(index_name)

print(f"✅ Pinecone index '{index_name}' ready — region: {PINECONE_ENV}")


✅ Pinecone index 'youtube-chunks' ready — region: us-east-1


/Users/sofiazogkza/repos/Ironhack/Ironhack Final Project/venvSofia/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 3: Embed the chunks from the JSON dataset and Upsert them into Pinecone using OpenAI embeddings

In [16]:
import json
import os
import openai
from time import sleep
from pinecone import Pinecone, ServerlessSpec  # new import

openai.api_key = os.getenv("OPENAI_API_KEY")

# Load dataset
with open("../output/rag_dataset_with_timestamps.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Function to get embeddings
def get_embedding(text, model="text-embedding-3-small"):
    for _ in range(3):
        try:
            return openai.Embedding.create(
                input=text,
                model=model
            )["data"][0]["embedding"]
        except Exception as e:
            print(f"Embedding error: {e}. Retrying...")
            sleep(1)
    return None

# --- Pinecone setup ---
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "youtube-subtitles"
dimension = 1536  # depends on embedding model, e.g., text-embedding-3-small
metric = "cosine"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-west1-gcp"
        )
    )

index = pc.index(index_name)

# --- Upsert embeddings ---
batch_size = 50
vectors = []

for i, chunk in enumerate(dataset):
    emb = get_embedding(chunk["text_chunk"])
    if emb is None:
        continue

    vectors.append((
        str(i),  # unique ID
        emb,
        {
            "video_title": chunk.get("video_title", ""),
            "url": chunk.get("url", ""),
            "start_time": chunk.get("start_time", ""),
            "end_time": chunk.get("end_time", "")
        }
    ))

    if len(vectors) >= batch_size:
        index.upsert(vectors=vectors)
        print(f"✅ Upserted batch ending with chunk {i}")
        vectors = []

# Upsert remaining vectors
if vectors:
    index.upsert(vectors=vectors)
    print(f"✅ Upserted final batch with {len(vectors)} chunks")

print("🎉 All chunks embedded and stored in Pinecone!")


NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-10', 'x-cloud-trace-context': '4f2f95c8ce0673b1ba90b9643b594688', 'date': 'Mon, 01 Dec 2025 22:08:20 GMT', 'server': 'Google Frontend', 'Content-Length': '106', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource cloud: aws region: us-west1-gcp not found"},"status":404}
